<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 80 Parity

The notebook works through the page as a pupil of 2002 worked through 3D-Baukasten 1.3, and checks
that every step does what it did then: the mouse marks a body point by point in two views, the
fields turn from yellow to green, Einfügen inserts behind the selected line, a CSG group gathers the
bodies up to its CSG-Ende, `#Neu` and `Neu_Ende` make a new body, the tool bar and the tabs Ausgabe,
Kamera, Einstellung and Bilder do their work, and the colour and the texture reach the body. Then it
walks the parity inventory `tests/fixtures/parity.json` through both looks: every element of 1.3
exists once at its place with its order and its text, the looks share one DOM, nothing of 1.3 is
displaced, and the new behaviour sits in places 1.3 left empty. Section 19 writes the report.

A check names the ID of the row of `Documentation/quirks.md` it proves, and a code cell carries the
tag of the column Test. A check that `expect` makes waits for the last cell when it fails. The
parameter `look` runs sections 1 to 12 in the other look, and `write_docs` writes
`Documentation/parity.md` from the inventory.

In [ ]:
width = 1440
height = 900
look = 'modern'
write_docs = False

In [ ]:
import sys, json, math, re
sys.path.insert(0, '..')
from harness import *

In [ ]:
site = Site()
page = Browser(site.url, width=width, height=height, look=look)
page.open()
DEFERRED = []


def expect(condition, message, *context):
    """Checks as check() does, and keeps a failure for the last cell instead of stopping here."""
    try:
        check(condition, message, *context)
    except Failure as failure:
        DEFERRED.append(str(failure))
        print(f"FAIL {message}")


def fresh(look=None):
    """Empties the scene through Datei › Neu and answers Ja, as a user does."""
    page.eval("() => { bausteine.act('neu'); }")
    page.settle()
    if page.eval("() => document.querySelector('#msgbox').open"):
        page.page.click('#msgbox .c-default')
        page.settle()
    page.key('Escape')
    page.key('Escape')


check(page.errors() == [], 'the page starts without an error', page.errors())


def at(view, x, y, z):
    """Answers the pixel of the page that shows the point (x, y, z) in the view `view`."""
    return page.eval(
        "([v, x, y, z]) => { const r = document.querySelector(`.view[data-view='${v}']`).getBoundingClientRect();"
        " const b = bausteine.app.bridge; const p = b.project(v, x, y, z); if (!p) return null;"
        " const w = b.cameras ? b.cameras[v * 18 + 16] : 0; const inset = w > 0 ? Math.max(0, (r.width - w) / 2) : 0;"
        " return [r.left + inset + p[0], r.top + inset + p[1]]; }", [view, x, y, z])


def hover_world(view, x, y, z):
    """Moves the pointer over the point (x, y, z) of the view `view`."""
    p = at(view, x, y, z)
    page.page.mouse.move(p[0], p[1])
    page.settle()
    return p

## 1. Datei › Neu

`mnuFileNew_Click` asks "Wollen Sie wirklich alle Objekte entfernen?" under the title FRAGE and
empties the scene on Ja. The raster is no item of the scene, so the list keeps `0 Raster`, and
Rückgängig brings the bodies back. Speichern keeps the meaning of 1.3 and writes the POV file.

In [ ]:
count = page.eval("() => bausteine.app.scene.items.length")
check(count > 0, 'the start scene holds bodies', count)
page.eval("() => { bausteine.act('neu'); }")
page.settle()
check(page.texts('#msgbox-title') == ['FRAGE'] and 'alle Objekte entfernen' in page.texts('#msgbox-text')[0], 'Q-7 Neu asks the question of 1.3')
page.page.click('#msgbox .c-default')
page.settle()
check(page.lines() == ['0 Raster'] and page.eval("() => bausteine.app.scene.items.length") == 0, 'F4-Q46 Ja replaces the items by none, and the list keeps 0 Raster, which is no item', page.lines())
page.key('Control+z'); page.settle()
check(page.eval("() => bausteine.app.scene.items.length") == count, 'Q-7 Rückgängig undoes Neu', page.eval("() => bausteine.app.scene.items.length"))
with page.page.expect_download() as download:
    page.eval("() => { bausteine.act('speichern'); }")
check(download.value.suggested_filename.endswith('.pov'), 'Q-7 Speichern writes the POV file as in 1.3', download.value.suggested_filename)
fresh()
check(page.lines() == ['0 Raster'], 'the scene is empty again', page.lines())

## 2. Marking a body with the mouse

A click on Quader in the body bar brings the tab Körper to the front, sets CB1 and starts the
handshake. A click in the Grundriss gives x and y, which turn green, and asks for z. A click in the
Aufriss gives z and keeps the y of the Grundriss, as the thesis describes it, frees the fields and
asks for the size. Two more clicks give the size as the signed difference to the point, and Einfügen
inserts the body. Between the clicks the preview follows the pointer, while the fields and their
locks change only with a click. The marks live on an overlay of the page that is drawn whole each
time and costs no request to the worker. The snap step is one unit, Alt takes a tenth and Shift no
snap in the modern look.

In [ ]:
OVERLAY = "() => Object.fromEntries([...document.querySelectorAll('.view')].map((v) => { const c = v.querySelector('canvas.view-overlay'); const d = c.getContext('2d').getImageData(0, 0, c.width, c.height).data; let n = 0; for (let i = 3; i < d.length; i += 4) if (d[i] > 0) n++; return [v.dataset.view, n]; }))"
idle = page.eval(OVERLAY)
page.graphql_log()
page.page.click('#tb2 [data-body="Quader"]')
page.settle()
check(page.eval("() => document.querySelector('#cb1-edit').value") == 'Quader' and page.eval("() => document.querySelector('#f4').dataset.tab") == 'koerper', 'Quader in CB1, the tab Körper in front')
page.graphql_log()
page.click_world(TOP, 2, 3, 0)
check(page.fields()[:3] == ['2*', '3*', '0'] and page.status() == 'z-Koordinate markieren', 'Q-47 the Grundriss gives x and y, which turn green, and asks for z', page.fields(), page.status())
colours = [page.css('#t1-1', 'color'), page.css('#t1-3', 'color')]
check(colours[0] != colours[1] and page.eval("() => [document.querySelector('#t1-1').classList.contains('locked'), document.querySelector('#t1-3').classList.contains('free')]") == [True, True], 'Q-47 the locked field is green and the free one yellow, kept in the data of the page', colours)
marked = page.eval(OVERLAY)
check(marked[str(TOP)] > idle[str(TOP)], 'Q-46 the mark is drawn on the overlay of the Grundriss', idle, marked)
page.click_world(FRONT, 0, 5, 1)
check(page.fields()[:3] == ['2', '3', '1'] and page.status() == 'Grösse angeben oder markieren', 'Q-45 the Aufriss gives z, keeps the y of the Grundriss and asks for the size', page.fields(), page.status())
check(page.graphql_log() == [], 'R-55 the marks cost no request to the worker')
page.click_world(TOP, 0, 1, 0)
check(page.fields()[3:5] == ['-2*', '-2*'], 'Q-43 a size towards the origin stays a signed difference', page.fields())
page.key('Escape'); page.settle()
check(page.eval(OVERLAY) == idle, 'Q-46 Esc clears the marks of every view, since the overlay is drawn whole each time', page.eval(OVERLAY))
page.page.click('#tb2 [data-body="Quader"]'); page.settle()
page.click_world(TOP, 2, 3, 0); page.click_world(FRONT, 0, 3, 1)
page.click_world(SIDE, 5, 0, 4)
check(page.fields()[3:6] == ['3*', '1', '3*'], 'Q-41 the Kreuzriss writes the size along x and the height', page.fields())
page.key('Escape')
grund, auf = page.rect(f'.view[data-view="{TOP}"]'), page.rect(f'.view[data-view="{FRONT}"]')
# The spy keeps every preview the page sends to the worker, the packed record of the body with its position at 2 to 4 and its size at 5 to 7.
SPY = "() => { const b = bausteine.app.bridge; const state = Object.getPrototypeOf(b).state; b.sent = []; b.state = (f) => { if ('preview' in f) b.sent.push(f.preview ? Array.from(f.preview) : null); return state.call(b, f); }; }"
sent = lambda: page.eval("() => bausteine.app.bridge.sent")
locked = lambda: page.eval("() => bausteine.mark().locked")
page.eval(SPY)
page.page.click('#tb2 [data-body="Quader"]'); page.settle()
page.click_world(TOP, 2, 3, 0)
still = page.sample_box(*auf)
hover_world(FRONT, 0, 3, 4)
check(page.sample_box(*auf) != still and page.fields()[:3] == ['2*', '3*', '0'] and locked() == [1, 2], 'Q-49 after the first mark the preview follows the pointer over the Aufriss, while the fields and their locks wait for the click', page.fields(), locked())
check(sent()[-1][2:8] == [2, 3, 4, 1, 1, 1], 'Q-49 the preview of the hover stands at the height under the pointer with the size of the fields', sent()[-1][2:8])
page.click_world(FRONT, 0, 3, 1)
still = page.sample_box(*grund)
hover_world(TOP, 6, 7, 0)
check(page.sample_box(*grund) != still and page.fields()[3:6] == ['1', '1', '1'] and locked() == [], 'Q-49 in phase 2 the preview follows the pointer over the Grundriss, while the size waits for the click', page.fields(), locked())
check(sent()[-1][2:8] == [2, 3, 1, 4, 4, 1], 'Q-49 the preview of the hover spans from the marked point to the point under the pointer', sent()[-1][2:8])
count = len(sent())
page.page.mouse.move(*at(TOP, 6.2, 7.2, 0)); page.settle()
check(len(sent()) == count, 'Q-49 a hover that snaps to the same point sends no preview', count, len(sent()))
page.page.mouse.move(5, 5); page.settle()
check(sent()[-1][2:8] == [2, 3, 1, 1, 1, 1] and page.sample_box(*grund) == still, 'Q-49 outside the views the preview follows the fields again', sent()[-1][2:8])
page.key('Escape')
page.eval("() => { const b = bausteine.app.bridge; delete b.state; delete b.sent; }")
page.page.click('#tb2 [data-body="Quader"]'); page.settle()
page.click_world(TOP, 2, 3, 0)
page.click_world(FRONT, 0, 3, 1)
before = page.sample_box(*grund)
page.page.fill('#t1-4', '3'); page.settle()
check(page.sample_box(*grund) != before, 'Q-49 a preview follows the fields once the point is marked', page.fields())
page.click_world(TOP, 6, 7, 0)
check(page.fields()[3:5] == ['4*', '4*'] and page.status() == 'Höhe markieren', 'the size in the Grundriss, then the height', page.fields(), page.status())
page.click_world(FRONT, 0, 7, 3)
check(page.fields()[3:6] == ['4', '4', '2'], 'the height in the Aufriss', page.fields())
before = page.sample_box(*grund)
page.page.fill('#t1-4', '5'); page.settle()
expect(page.sample_box(*grund) != before, 'Q-49 the preview stays after the last mark and follows a typed size until Einfügen', page.fields())
page.page.fill('#t1-4', '4'); page.settle()
page.page.click('#btn-einf')
page.settle()
items = page.eval("() => bausteine.app.scene.items.map((i) => [i.kind, i.position, i.size])")
check(items == [['CUBOID', {'x': 2, 'y': 3, 'z': 1}, {'x': 4, 'y': 4, 'z': 2}]], 'Q-49 the marks do not insert the body, Einfügen inserts the marked Quader', items)
check(page.lines() == ['0 Raster', '1 Quader'] and page.status().startswith('Eingefügt'), 'the list and the status', page.lines())
page.page.click('#tb2 [data-body="Kugel"]'); page.settle()
p = at(TOP, 2.34, 3.27, 0)
if look == 'modern':
    page.page.keyboard.down('Alt'); page.page.mouse.click(p[0], p[1]); page.page.keyboard.up('Alt'); page.settle()
    tenth = page.fields()[:2]
    page.key('Escape'); page.page.click('#tb2 [data-body="Kugel"]'); page.settle()
    page.page.keyboard.down('Shift'); page.page.mouse.click(p[0], p[1]); page.page.keyboard.up('Shift'); page.settle()
    free = page.fields()[:2]
    page.key('Escape'); page.page.click('#tb2 [data-body="Kugel"]'); page.settle()
    page.page.mouse.click(p[0], p[1]); page.settle()
    check(page.fields()[:2] == ['2*', '3*'] and tenth == ['2,3*', '3,3*'] and free == ['2,34*', '3,27*'], 'Q-27 the snap step is one unit, Alt takes a tenth and Shift no snap', page.fields()[:2], tenth, free)
else:
    page.page.keyboard.down('Alt'); page.page.mouse.click(p[0], p[1]); page.page.keyboard.up('Alt'); page.settle()
    check(page.fields()[:2] == ['2*', '3*'], 'Q-27 the classic look takes the step of the setting alone', page.fields()[:2])
page.key('Escape'); page.key('Escape')

## 3. A group of the CSG

CSG-Differenz opens a group, the next bodies go into it, CSG-Ende closes it, and the body after it
stands behind the group, as the flat list of 1.3 has it. The mask of a group offers only what a
group has.

In [ ]:
page.entry('CSG-Ende')
check(page.visible('#geo-hint') and not any(page.visible(f'#t1-{i}') for i in range(1, 13)), 'F4-Q38 the mask of CSG-Ende offers no field, only its hint', page.texts('#geo-hint'))
page.entry('CSG-Differenz')
check(page.texts('#l1-1, #l1-4, #l1-7') == ['Verschiebung der Glieder (x,y,z)', 'Skalierung x y z', 'Drehung um X,Y,Z - Achse'] and not page.visible('#t1-10') and page.visible('#geo-hint'), 'F4-Q38 the mask of a group offers only what a group has, the move, the scale and the turn of its members', page.texts('#l1-1, #l1-4, #l1-7'))
page.page.click('#btn-einf'); page.settle()
check(not any('unbekannt' in t.lower() for t in [page.status()]), 'F4-Q38 and Einfügen prints nothing about an unknown body', page.status())
page.page.click('#tb2 [data-body="Würfel"]'); page.page.click('#btn-einf'); page.settle()
page.page.click('#tb2 [data-body="Zylinder"]'); page.page.click('#btn-einf'); page.settle()
page.entry('CSG-Ende'); page.page.click('#btn-einf'); page.settle()
page.page.click('#tb2 [data-body="Kugel"]'); page.page.click('#btn-einf'); page.settle()
check(page.lines() == ['0 Raster', '1 Quader', '2 CSG-Differenz', '3   -Würfel', '4   -Zylinder', '5 CSG-Ende', '6 Kugel'], 'the lines of 1.3', page.lines())

## 4. A new body

`#Neu` starts a new body, named Werkstück1 as in the thesis, whose name the field Name edits,
`Neu_Ende` closes it, and its name stands in CB1 from then on, from where it is placed again. The
mask of a placed body reads `Bezugspunkt (x,y,z)` and `Skalierung x y z`.

In [ ]:
page.entry('#Neu'); page.page.click('#btn-einf'); page.settle()
check(page.eval("() => document.querySelector('#t1-11').value") == 'Werkstück1' and page.visible('#t1-11'), 'F4-Q48 #Neu inserts a declaration named Werkstück1, its name in the field Name', page.eval("() => document.querySelector('#t1-11').value"))
page.page.click('#tb2 [data-body="Kegel"]'); page.page.click('#btn-einf'); page.settle()
page.entry('Neu_Ende'); page.page.click('#btn-einf'); page.settle()
names = page.eval("() => [...document.querySelectorAll('#cb1 .c-combo-list [data-entry]')].map((e) => e.dataset.entry)")
check(names[-1] == 'Werkstück1', 'the new body is an entry of CB1', names[-3:])
page.entry('Werkstück1')
check(page.texts('#l1-1') == ['Bezugspunkt (x,y,z)'] and page.texts('#l1-4') == ['Skalierung x y z'], 'F4-Q60 the mask of an instance reads Bezugspunkt and Skalierung', page.texts('#l1-1'), page.texts('#l1-4'))
page.page.click('#btn-einf'); page.settle()
lines = page.lines()
check(lines[7:] == ['7 #Neu Werkstück1', '8 Kegel', '9 Werkstück1', '10 Werkstück1'], 'declared, ended and placed again', lines)
kinds = page.eval("() => bausteine.app.scene.items.map((i) => i.kind)")
check(kinds.count('DECLARE') == 1 and kinds.count('INSTANCE') == 1, 'one declaration and one instance', kinds)

## 5. Ändern and Löschen

A line of the list shows its body in the mask, Ändern applies the fields and keeps the selection,
Löschen removes the body.

In [ ]:
page.page.click('#cb2 [data-line="1"]'); page.settle()
check(page.fields()[:3] == ['2', '3', '1'], 'the Quader in the mask', page.fields())
page.page.fill('#t1-3', '2')
page.page.click('#btn-change'); page.settle()
quader = page.eval("() => bausteine.app.scene.items.find((i) => i.kind === 'CUBOID').position")
check(quader == {'x': 2, 'y': 3, 'z': 2} and page.eval("() => bausteine.app.line") == 1, 'F4-Q36 Ändern moves it and keeps the selection', quader)
page.page.click('#btn-clear'); page.settle()
check(page.eval("() => bausteine.app.scene.items.some((i) => i.kind === 'CUBOID')") is False, 'Löschen removes it')
page.page.keyboard.press('Control+z'); page.settle()
check(page.eval("() => bausteine.app.scene.items.some((i) => i.kind === 'CUBOID')") is True, 'Strg+Z brings it back')

## 6. The tool bar

Position hides the body bar, Körper einfügen shows it again and brings tab Körper to the front, and
the active mode shows pressed while the other buttons do not latch. 3D-Zoom shows the Schrägriss
over all four fields, centred and scaled, and keeps it through a resize of the window. Dia-Show
brings the tab Bilder to the front, Raytrace starts the ray tracer of the Schrägriss in the browser
and shows the command line of 1.3 on tab Ausgabe.

In [ ]:
pressed = lambda: page.texts('#tb1 [data-tool="1"], #tb1 [data-tool="2"], #tb1 [data-tool="3"]', 'aria-pressed')
page.page.click('#tb1 [data-tool="1"]'); page.settle()
check(not page.visible('#f2') and page.eval("() => bausteine.app.mode") == 'position', 'Q-79 Position hides F2')
check(pressed() == ['true', 'false', 'false'], 'Q-64 the active mode Position shows pressed', pressed())
page.page.click('#sstab1 [data-tab="kamera"]'); page.settle()
page.page.click('#tb1 [data-tool="2"]'); page.settle()
check(page.visible('#f2') and page.eval("() => bausteine.app.mode") == 'einfuegen' and page.eval("() => document.querySelector('#f4').dataset.tab") == 'koerper', 'Q-79 Körper einfügen shows F2 and brings tab Körper to the front')
check(pressed() == ['false', 'true', 'false'], 'Q-64 and shows pressed in its place', pressed())
page.page.click('#tb1 [data-tool="3"]'); page.settle()
check(pressed() == ['false', 'false', 'true'] and page.eval("() => bausteine.app.mode") == 'bearbeiten', 'Q-64 Bearbeiten is a mode of its own')
page.page.click('#tb1 [data-tool="4"]'); page.settle()
check(page.texts('#tb1 [data-tool="4"]', 'aria-pressed') in ([None], ['false']), 'Q-64 the other buttons do not latch')
page.key('Escape'); page.page.click('#tb1 [data-tool="1"]'); page.settle()
page.page.click('#tb1 [data-tool="6"]'); page.settle()
oblique, area = page.rect('.view[data-view="3"]'), page.rect('#views')
check(page.visible('.view[data-view="3"]') and not any(page.visible(f'.view[data-view="{n}"]') for n in (0, 1, 2)), 'Q-57 3D-Zoom hides the three normal views')
check(abs(oblique[2] - area[2]) <= 2 and abs(oblique[3] - area[3]) <= 2, 'Q-57 and the Schrägriss fills the four fields', oblique, area)
page.page.set_viewport_size({'width': width - 200, 'height': height - 100}); page.settle()
check(page.visible('.view[data-view="3"]') and not page.visible('.view[data-view="1"]') and abs(page.rect('.view[data-view="3"]')[2] - page.rect('#views')[2]) <= 2, 'Q-58 the toggle survives a resize', page.rect('.view[data-view="3"]'))
page.page.set_viewport_size({'width': width, 'height': height}); page.settle()
page.page.click('#tb1 [data-tool="6"]'); page.settle()
check(page.visible('.view[data-view="1"]'), 'a second click shows all four again')
page.page.click('#tb1 [data-tool="9"]'); page.settle()
check(page.eval("() => document.querySelector('#f4').dataset.tab") == 'bilder' and not page.visible('#f9'), 'Q-83 Dia-Show only brings the tab Bilder, the show starts with Anzeigen')
page.page.click('#tb1 [data-tool="5"]'); page.settle()
check(page.eval("() => bausteine.app.raytrace") is True and page.texts('#tb1 [data-tool="5"]', 'aria-pressed') == ['true'], 'R-58 Raytrace starts the ray tracer of the Schrägriss in the browser')
check('pvengine.exe' in page.texts('#pov-command')[0], 'R-58 and tab Ausgabe shows the command line of 1.3', page.texts('#pov-command'))
page.page.click('#tb1 [data-tool="5"]'); page.settle()

## 7. The 3D zoom and the 2D zoom

The buttons `<` zoom in and `>` zoom out, the 2D zoom on the ladder of 1.3 from 12 by 1.2, the 3D
zoom on its own ladder, and each pair zooms its own views only. One 3D zoom serves the buttons, a
click into the Schrägriss, the menu and the slider of tab Kamera, and it survives every edit. No
timer rewrites the field of the zoom, which changes when the zoom changes. A drag in the Schrägriss
orbits in the modern look.

In [ ]:
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
while abs(page.eval("() => bausteine.app.zoomf") - 12) > 1e-6:
    page.page.click('#d2-zoomin' if page.eval("() => bausteine.app.zoomf") < 12 else '#d2-zoomout'); page.settle()
before = page.eval("() => [bausteine.app.zoomf, bausteine.app.d3zoomf]")
page.page.click('#d2-zoomin')
check(page.eval("() => document.querySelector('#tzoom').value") == zoom_text(14.4, look), 'Q-3 the field of the zoom changes when the zoom changes', page.eval("() => document.querySelector('#tzoom').value"))
check(abs(page.eval("() => bausteine.app.zoomf") - 14.4) < 1e-9 and page.eval("() => bausteine.app.d3zoomf") == before[1], 'Q-52 < zooms 2D in by 1.2 and leaves the 3D zoom', page.eval("() => [bausteine.app.zoomf, bausteine.app.d3zoomf]"))
page.page.click('#d3-zoomin'); page.settle()
check(abs(page.eval("() => bausteine.app.d3zoomf") - before[1] * 1.2) < 1e-9 and abs(page.eval("() => bausteine.app.zoomf") - 14.4) < 1e-9, 'Q-52 the 3D buttons zoom the 3D view only', page.eval("() => [bausteine.app.zoomf, bausteine.app.d3zoomf]"))
page.page.click('#d3-zoomout'); page.settle()
text = page.eval("() => document.querySelector('#tzoom').value")
page.eval("() => { bausteine.app.zoomf = 30; }"); page.settle()
# A check that nothing follows lets the time of several ticks of Timer1 of 1.3 pass.
page.page.wait_for_timeout(400)
check(page.eval("() => document.querySelector('#tzoom').value") == text, 'Q-3 no timer polls the zoom into the field')
page.eval("() => { bausteine.app.zoomf = 14.4; }")
d = page.eval("() => bausteine.app.d3zoomf")
page.click_world(OBLIQUE, 0, 0, 0)
after = page.eval("() => [bausteine.app.zoomf, bausteine.app.d3zoomf]")
check(after[0] == 14.4 and abs(after[1] - d * 1.2) < 1e-9, 'Q-55 a click in the Schrägriss zooms the same 3D zoom', after)
page.menu('ansicht/zoom3d-in')
check(abs(page.eval("() => bausteine.app.d3zoomf") - d * 1.44) < 1e-9, 'Q-55 and so does the menu', page.eval("() => bausteine.app.d3zoomf"))
page.page.click('#sstab1 [data-tab="kamera"]'); page.settle()
check(page.eval("() => document.querySelector('#hs-4').value") == str(round(d * 14.4)), 'Q-55 the slider of tab Kamera shows the same zoom', page.eval("() => document.querySelector('#hs-4').value"))
page.page.fill('#hst-4', '100'); page.page.press('#hst-4', 'Enter'); page.settle()
check(abs(page.eval("() => bausteine.app.d3zoomf") - 10) < 1e-9, 'Q-55 and sets it', page.eval("() => bausteine.app.d3zoomf"))
page.page.click('#d3-zoomin'); page.settle()
zoom3d = page.eval("() => bausteine.app.d3zoomf")
page.page.click('#tb1 [data-tool="2"]'); page.settle()
page.page.click('#tb2 [data-body="Kugel"]'); page.page.click('#btn-einf'); page.settle()
page.page.click('#cb2 [data-line="1"]'); page.settle()
page.page.fill('#t1-1', '3'); page.page.click('#btn-change'); page.settle()
check(page.eval("() => bausteine.app.d3zoomf") == zoom3d, 'Q-53 the 3D zoom survives an insert and an edit', zoom3d, page.eval("() => bausteine.app.d3zoomf"))
page.key('Control+z'); page.settle(); page.key('Control+z'); page.settle()
if look == 'modern':
    orbit = page.eval("() => JSON.stringify(bausteine.app.orbit)")
    r = page.rect('.view[data-view="3"]')
    page.page.mouse.move(r[0] + r[2] / 2, r[1] + r[3] / 2); page.page.mouse.down()
    page.page.mouse.move(r[0] + r[2] / 2 + 60, r[1] + r[3] / 2 + 20, steps=6); page.page.mouse.up(); page.settle()
    check(page.eval("() => JSON.stringify(bausteine.app.orbit)") != orbit and page.eval("() => bausteine.app.d3zoomf") == zoom3d, 'Q-52 a drag in the Schrägriss orbits in the modern look', orbit)
page.page.click('#d3-zoomout'); page.settle()

## 8. Ausgabe

Übersetzen writes the text for POV-Ray into Text1 and enables Ausgeben, and leaves the selection
alone. The text stays editable until the next Übersetzen. Ausgeben offers a file named after the
settings, keeps a typed `.pov` single, confirms the download in the status line and is disabled
again. Tab Ausgabe shows the command line of 1.3 built from the paths of tab Einstellung.

In [ ]:
page.page.click('#cb2 [data-line="1"]'); page.settle()
selected = page.eval("() => bausteine.app.line")
page.page.click('#sstab1 [data-tab="ausgabe"]'); page.settle()
page.page.click('#bpov-comp'); page.settle()
text = page.eval("() => document.querySelector('#text1').value")
check(text.startswith('// ====' if look == 'modern' else '//====') and '#include "colors.inc"' in text and 'difference {' in text, 'Übersetzen writes the scene', text[:120])
check(page.eval("() => bausteine.app.line") == selected, 'M-32 the export reads the scene and leaves the selection alone', selected)
check(page.eval("() => document.querySelector('#bpov-save').disabled") is False, 'F4-Q69 Ausgeben is enabled')
page.page.fill('#text1', text + '// von Hand')
check(page.eval("() => document.querySelector('#text1').value").endswith('// von Hand'), 'F4-Q68 the text stays editable')
with page.page.expect_download() as download:
    page.page.click('#bpov-save')
check(download.value.suggested_filename == 'ausgabe.pov' and open(download.value.path(), encoding='utf-8').read().endswith('// von Hand'), 'T-p95d Ausgeben offers ausgabe.pov named after the settings, with the text as it stands', download.value.suggested_filename)
check(page.eval("() => document.querySelector('#bpov-save').disabled") is True and page.status() == 'ausgabe.pov gespeichert', 'F4-Q69 the status confirms the download, and the button is disabled again', page.status())
page.page.click('#bpov-comp'); page.settle()
check(not page.eval("() => document.querySelector('#text1').value").endswith('// von Hand'), 'F4-Q68 the next Übersetzen writes the text anew')
name = page.eval("() => document.querySelector('#tfname').value")
page.page.fill('#tfname', 'C:\\3DBAU\\haus.pov')
with page.page.expect_download() as download:
    page.page.click('#bpov-save')
check(download.value.suggested_filename == 'haus.pov', 'F4-Q70 a typed .pov is not doubled', download.value.suggested_filename)
page.page.fill('#tfname', name)
command = page.texts('#pov-command')[0]
check('pvengine.exe +W400 +H300' in command and 'C:\\3DBAU' in command, 'F4-Q74 the command line of 1.3 for POV-Ray, built from the paths of tab Einstellung', command)

## 9. Kamera

The fields show the values of the sliders from the start and refuse text with a status line, a typed
number moves its slider clamped to its range, and the slider of the zoom shows the 3D zoom in
percent, for which F5 has no field. The sliders give the degrees still to turn about x, y and z,
Start turns the Schrägriss by 2 degrees every 100 ms in the sense of the sign, only while it turns,
and Stopp stops it. The turn changes the display only. Einfügen takes the camera of the Schrägriss
into the export, Löschen gives back the camera of 1.3, and the light, the sky and the floor stay
those of 1.3.

In [ ]:
page.page.click('#sstab1 [data-tab="kamera"]'); page.settle()
values = page.eval("() => ['#hst-4', '#hst-1', '#hst-2', '#hst-3'].map((s) => document.querySelector(s).value)")
check(values[1:] == ['0', '0', '0'] and values[0] == page.eval("() => document.querySelector('#hs-4').value"), 'F4-Q9 the fields show the values of the sliders', values)
check(page.eval("() => document.querySelectorAll('#f5 input').length") == 4, 'T-p90h F5 has no field of the 3D zoom, the slider shows it in percent')
page.page.fill('#hst-1', 'abc'); page.page.press('#hst-1', 'Enter'); page.settle()
check(page.status().startswith('Bitte eine Zahl') and page.eval("() => document.querySelector('#hst-1').value") == '0', 'F4-Q9 text is refused with a status line', page.status())
page.page.fill('#hst-1', '400'); page.page.press('#hst-1', 'Enter'); page.settle()
check(page.eval("() => document.querySelector('#hs-1').value") == '360', 'F4-Q9 a typed number moves its slider, clamped to its range')
page.page.fill('#hst-1', '0'); page.page.press('#hst-1', 'Enter')
page.page.fill('#hst-4', '150'); page.page.press('#hst-4', 'Enter'); page.settle()
check(abs(page.eval("() => bausteine.app.d3zoomf") - 15) < 1e-9, 'F4-Q7 the slider sets the 3D zoom in percent', page.eval("() => bausteine.app.d3zoomf"))
page.page.click('#d3-zoomin'); page.settle()
check(page.eval("() => document.querySelector('#hs-4').value") == '180', 'F4-Q7 and follows it', page.eval("() => document.querySelector('#hs-4').value"))
page.page.fill('#hst-4', '100'); page.page.press('#hst-4', 'Enter'); page.settle()
rotations = page.eval("() => JSON.stringify(bausteine.app.scene.items.map((i) => i.rotation))")
TURNED = "() => document.querySelector('#start').getAttribute('aria-pressed') === 'false'"
page.page.fill('#hst-3', '20'); page.page.press('#hst-3', 'Enter')
page.page.click('#start'); page.until(TURNED)
turn = page.eval("() => bausteine.app.turn")
check(abs(turn['z'] - 20) < 1e-9 and page.eval("() => document.querySelector('#hs-3').value") == '0', 'F4-Q89 Start turns by the value of the slider', turn)
check(page.texts('#start', 'aria-pressed') == ['false'], 'F4-Q87 the step of 100 ms ends with the turn')
check(page.eval("() => JSON.stringify(bausteine.app.scene.items.map((i) => i.rotation))") == rotations, 'T-p79a the turn changes the display only, the angles of the bodies stay')
page.page.fill('#hst-2', '-6'); page.page.press('#hst-2', 'Enter')
page.page.click('#start'); page.until(TURNED)
check(abs(page.eval("() => bausteine.app.turn.y") + 6) < 1e-9, 'F4-Q89 a negative value turns the other way', page.eval("() => bausteine.app.turn"))
page.page.fill('#hst-1', '90'); page.page.press('#hst-1', 'Enter')
page.page.click('#start'); page.until("() => bausteine.app.turn.x > 0")
page.page.click('#command2'); page.settle()
stopped = page.eval("() => bausteine.app.turn.x")
# A check that no step follows lets the time of five steps of 100 ms pass.
page.page.wait_for_timeout(500)
check(0 < stopped < 90 and page.eval("() => bausteine.app.turn.x") == stopped, 'F4-Q6 Stopp stops the turn, F4-Q87 and no step follows', stopped)
page.page.dblclick('#command2'); page.settle()
page.page.click('#btn1-1'); page.settle()
povtext = page.eval("() => document.querySelector('#text1').value")
check('camera { location <15, 8, -15>' not in povtext, 'F4-Q5 Einfügen takes the camera of the Schrägriss', povtext[povtext.find('camera'):][:80])
check('light_source' in povtext and 'plane' in povtext, 'F4-Q92 the light and the floor stay those of 1.3')
page.page.click('#btn2-1'); page.settle()
check('location <15, 8, -15>' in page.eval("() => document.querySelector('#text1').value"), 'F4-Q5 Löschen gives back the camera of 1.3')

## 10. Einstellung

The paths stay locked until the password 2002 of 1.3, Übernehmen stores the four paths, rebuilds the
file name of tab Ausgabe and locks again, and Standard restores the defaults and locks again too.
The button Bildschirmeinstellungen opens the dialog of 2.0 with the look, the dark theme, the
virtual screen, the inverted views, the snap step and the colour of the frame GEOMETRIEDATEN.

In [ ]:
page.page.click('#sstab1 [data-tab="einstellung"]'); page.settle()
check(page.eval("() => document.querySelector('#tbau-home').disabled") is True, 'the paths are locked')
page.page.fill('#tpasswd', '2002')
check(page.eval("() => document.querySelector('#tbau-home').disabled") is False, 'F4-Q82 the same password 2002 unlocks them')
page.page.fill('#tbsp-def', 'haus')
page.page.click('#bdatei-change'); page.settle()
check(page.eval("() => document.querySelector('#tfname').value").endswith('haus') and page.eval("() => document.querySelector('#tbau-home').disabled") is True, 'F4-Q83 Übernehmen rebuilds the file name and locks again')
check(json.loads(page.eval("() => localStorage.getItem('bausteine.einstellung')"))['bspDef'] == 'haus', 'F4-Q83 and stores the paths')
page.page.fill('#tpasswd', '2002'); page.page.click('#bdatei-def'); page.settle()
check(page.eval("() => document.querySelector('#tbau-home').disabled") is True and page.eval("() => document.querySelector('#tfname').value") == 'C:\\3DBAU\\ausgabe', 'F4-Q85 Standard restores the defaults and locks again as well')
check(page.eval("() => document.querySelector('#bscreen').disabled") is True, 'F4-Q86 Bildschirmeinstellungen waits for the password as well')
page.page.fill('#tpasswd', '2002')
page.page.click('#bscreen'); page.settle()
fields = page.eval("() => { const f = document.querySelector('#screen-dialog form'); return [...new Set([...f.elements].map((e) => e.name).filter(Boolean))]; }")
check(page.eval("() => document.querySelector('#screen-dialog').open") and fields == ['look', 'dark', 'inverted', 'screen', 'snap', 'geo'], 'F4-Q86 the button opens Bildschirmeinstellungen with the look, the dark theme, the screen, the inverted views, the snap and the frame colour', fields)
page.page.click('#screen-dialog button[value="cancel"]'); page.settle()

## 11. Colour and texture

The colour dialog of Windows has 48 basic and 16 custom colours and opens with the colour of the
body, OK gives the colour to the selected body, Abbrechen changes nothing, and the frame FARBE of F6
shows it. TEXTUR gives the selected body a texture of POV-Ray and is disabled while no body is
selected, `0 Raster` included.

In [ ]:
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
page.page.click('#cb2 [data-line="0"]'); page.settle()
check(page.eval("() => document.querySelector('#cb3').disabled") is True, 'M-64 TEXTUR is disabled while 0 Raster is selected')
page.page.click('#cb2 [data-line="1"]'); page.settle()
check(page.eval("() => document.querySelector('#cb3').disabled") is False, 'M-64 and enabled for a body')
colour = page.eval("() => bausteine.app.scene.items.find((i) => i.kind === 'CUBOID').color")
page.page.click('#bcolor'); page.settle()
rgb = page.eval("() => ['#farben-r', '#farben-g', '#farben-b'].map((s) => Number(document.querySelector(s).value))")
check('#%02x%02x%02x' % tuple(rgb) == colour.lower(), 'M-65 the dialog opens with the colour of the body', rgb, colour)
check(page.eval("() => document.querySelectorAll('.farben-grund .farben-swatch').length") == 48 and page.eval("() => document.querySelectorAll('.farben-custom .farben-swatch').length") == 16, '48 basic and 16 custom colours')
page.page.click('.farben-grund [data-color="#FF0000"]')
page.page.click('#farben-cancel'); page.settle()
check(page.eval("() => bausteine.app.scene.items.find((i) => i.kind === 'CUBOID').color") == colour, 'M-65 Abbrechen changes nothing')
page.page.click('#bcolor'); page.settle()
page.page.click('.farben-grund [data-color="#FF00FF"]')
page.page.click('#farben-ok'); page.settle()
quader = page.eval("() => bausteine.app.scene.items.find((i) => i.kind === 'CUBOID')")
check(quader['color'] == '#ff00ff', 'M-65 OK gives the colour to the body', quader['color'])
check(page.css('#frame-farbe', 'background-color') == '#ff00ff', 'M-66 the frame FARBE shows it')
page.page.select_option('#cb3', 'Chrome_Metal'); page.settle()
textures = page.eval("() => bausteine.app.scene.items.map((i) => i.texture)")
check(textures.count('Chrome_Metal') == 1 and page.eval("() => bausteine.app.scene.items.find((i) => i.kind === 'CUBOID').texture") == 'Chrome_Metal', 'M-64 TEXTUR gives the texture to the selected body only', textures)

## 12. Bilder and the Dia-Show

Ansicht › Bild speichern keeps a picture of a view in the list of the tab Bilder, which holds
pictures only, with the first one selected. TDatei shows the path built from the settings and is
read only. Anzeigen shows the picture black over the screen, the ends of the list end the show, End
closes it, and Beenden closes the show as well.

In [ ]:
kept = page.eval("() => document.querySelectorAll('#file1 [data-index]').length")
for _ in range(2):
    with page.page.expect_download():
        page.eval("() => { bausteine.act('bild-speichern'); }")
    page.settle()
page.page.click('#sstab1 [data-tab="bilder"]')
page.until("(n) => document.querySelectorAll('#file1 [data-index]').length >= n", kept + 2)
rows = page.eval("() => [...document.querySelectorAll('#file1 [data-index]')].map((r) => [r.textContent, r.getAttribute('aria-selected')])")
check(len(rows) >= 2, 'R-57 the pictures saved from the views are in the list the browser keeps', rows)
check(rows[0][1] == 'true' and [r[1] for r in rows].count('true') == 1, 'F4-Q77 the first picture is selected', rows)
check(all(r[0].lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp')) for r in rows), 'F4-Q78 the list holds pictures only', rows)
check(page.eval("() => document.querySelector('#tdatei').readOnly") and page.eval("() => document.querySelector('#tdatei').value").startswith('C:\\3DBAU\\'), 'F4-Q80 TDatei shows the path built from the settings and is read only', page.eval("() => document.querySelector('#tdatei').value"))
page.page.click('#bbild-anz')
page.until("() => { const i = document.querySelector('#f9-picture'); return !document.querySelector('#f9').hidden && i.complete && i.naturalWidth > 0; }")
check(page.visible('#f9') and page.eval("() => document.querySelector('#f9-picture').naturalWidth") > 0, 'T-p77a Anzeigen shows the picture')
page.page.click('#bbild-rueck'); page.settle()
check(not page.visible('#f9') and page.visible('#f2'), 'Q-102 < on the first picture ends the show, and the bars show again')
page.page.click('#bbild-anz'); page.settle()
page.page.click('#bbild-ende'); page.settle()
check(not page.visible('#f9'), 'End closes the show')
page.page.click('#bbild-anz'); page.settle()
page.eval("() => document.querySelector('#bbild-beenden').click()"); page.settle()
check(not page.visible('#f9'), 'F4-Q81 Beenden also closes the show')
check(page.errors() == [], 'no error on the way', page.errors())

## 13. The inventory is complete and valid

The parity inventory `tests/fixtures/parity.json` holds one record per element and behaviour of 1.3,
`P-001` to `P-230` of the parity plan with the record of F6 that the EXE holds, `P-200a`, and the
tool tip of CB3, `P-203b`. Every record has every field, the IDs are unique, every selector is valid
CSS, and every container is a record or a window.

In [ ]:
WINDOWS = ['#form1', '#f2', '#f3', '#f4', '#f5', '#f6', '#farben', '#f9']

#: Answers what one record of the inventory finds in the page: the box of
#: its element, its box relative to the client of its container, its place
#: among its group in reading order, and every problem with its place, its
#: order and its text.
WALK = """
([rec, look, screen]) => {
  const out = { problems: [] };
  const all = document.querySelectorAll(rec.selector);
  if (all.length !== 1) { out.problems.push(`${all.length} elements for ${rec.selector}`); return out; }
  const e = all[0];
  const r = e.getBoundingClientRect();
  out.rect = [r.left, r.top, r.width, r.height];
  if (!(r.width > 0 && r.height > 0)) out.problems.push('not visible');
  if (rec.container) {
    const c = document.querySelector(rec.container);
    if (!c) out.problems.push('no container');
    else {
      const k = c.getBoundingClientRect();
      const clipped = rec.clipped && look === 'classic' && rec.clipped.includes(screen);
      if (!clipped && (r.left < k.left - 2 || r.top < k.top - 2 || r.right > k.right + 2 || r.bottom > k.bottom + 2))
        out.problems.push(`outside its container ${rec.container}`);
      const client = (c.matches('#form1, #f2, #f3, #f4, #f5, #f6, #farben, #f9') && c.querySelector(':scope > .win-client')) || c;
      const q = client.getBoundingClientRect();
      out.rel = [r.left - q.left, r.top - q.top, r.width, r.height];
    }
  }
  if (rec.group) {
    const items = [...document.querySelectorAll(rec.group)].map((x) => ({ x, b: x.getBoundingClientRect() }))
      .filter((o) => o.b.width > 0 && o.b.height > 0);
    items.sort((a, b) => (a.b.top + a.b.height / 2) - (b.b.top + b.b.height / 2));
    const rows = [];
    for (const o of items) {
      const row = rows[rows.length - 1];
      if (row && o.b.top < row.bottom && o.b.bottom > row.top) {
        row.items.push(o); row.top = Math.min(row.top, o.b.top); row.bottom = Math.max(row.bottom, o.b.bottom);
      } else rows.push({ items: [o], top: o.b.top, bottom: o.b.bottom });
    }
    const ordered = rows.flatMap((row) => row.items.sort((a, b) => a.b.left - b.b.left)).map((o) => o.x);
    out.order = ordered.indexOf(e);
    if (out.order !== rec.order) out.problems.push(`order ${out.order}, not ${rec.order}`);
  }
  if (rec.text !== null) {
    const want = typeof rec.text === 'string' ? rec.text : rec.text[look];
    const norm = (s) => (s ?? '').replace(/\\s+/g, ' ').trim();
    const seen = [e.textContent, e.querySelector(':scope > legend')?.textContent, e.querySelector(':scope > span')?.textContent,
      e.dataset.tip, e.getAttribute('title'), e.getAttribute('aria-label'), e.value].map(norm);
    if (want !== undefined && !seen.includes(norm(want))) out.problems.push(`text ${JSON.stringify(want)} not in ${JSON.stringify(seen.filter(Boolean))}`);
  }
  return out;
}
"""

#: Closes every dialog, menu, list and card the walk may have opened.
RESET = """
() => {
  document.querySelectorAll('dialog[open]').forEach((d) => d.close());
  document.querySelectorAll('#menubar .menu-popup').forEach((p) => { if (!p.classList.contains('submenu-popup')) p.hidden = true; });
  document.querySelectorAll('#menubar .menu-title').forEach((t) => t.setAttribute('aria-expanded', 'false'));
  const list = document.querySelector('#cb1 .c-combo-list'); if (list) list.hidden = true;
  const s = document.querySelector('#splash'); if (s) s.hidden = true;
}
"""


def trigger(b, step):
    """Takes the step `step` of a record before its check, as the field trigger of the fixture says."""
    kind, _, arg = step.partition(':')
    if kind == 'menu':
        b.page.click(f'#menubar .menu-item[data-menu="{arg}"] > .menu-title')
    elif kind == 'list':
        b.page.click(arg)
    elif kind == 'entry':
        b.entry(arg)
    elif kind == 'act':
        b.eval('(n) => bausteine.act(n)', arg)
    elif kind == 'click':
        b.page.click(arg)
    elif kind == 'key':
        b.key(arg)
    b.settle()


def walk(b, look, screen=None, twips=False):
    """Walks every record of the inventory for the look `look` in the page `b` and answers the findings by ID."""
    results = {}
    for r in INVENTORY['records']:
        if look not in r['looks'] or r['checkedIn'] or r['kind'] in ('behaviour', 'absent', 'message'):
            continue
        b.page.keyboard.press('Escape')
        b.eval(RESET)
        if r['tab']:
            b.page.click(f'#sstab1 .tab[data-tab="{r["tab"]}"]')
            b.settle()
        if r['trigger']:
            try:
                trigger(b, r['trigger'])
            except Exception as error:  # noqa: BLE001, the record names the failure
                results[r['id']] = {'problems': [f'the trigger failed: {str(error)[:160]}']}
                continue
        res = {'problems': []}
        if r['kind'] in ('element', 'menu-entry', 'tooltip'):
            res = b.eval(WALK, [r, look, screen])
        if r['effect'] and not b.eval(f"() => Boolean({r['effect']})"):
            res['problems'].append('the effect does not hold')
        if r['kind'] == 'state' and r['text'] is not None and r['selector']:
            text = b.eval(WALK, [{**r, 'group': None, 'container': None}, look, screen])
            res['problems'] += [p for p in text['problems'] if p.startswith('text')]
        if twips and r['twips'] and 'rel' in res:
            want = [v / 15 for v in r['twips']]
            res['twips'] = [round(res['rel'][i] - want[i], 1) for i in range(4)]
        results[r['id']] = res
        if r['trigger'] and r['trigger'].startswith('entry:'):
            b.entry('Quader')
    b.page.keyboard.press('Escape')
    b.eval(RESET)
    return results


check(INVENTORY is not None and INVENTORY['SPDX-License-Identifier'] == 'GPL-2.0-only', 'the inventory is a file of 2.0 under GPL-2.0-only')
records = INVENTORY['records']
ids = [r['id'] for r in records]
check(len(ids) == len(set(ids)) == 196, 'the 196 records have unique IDs', len(ids))
check({'P-200a', 'P-203b'} <= set(ids) and all(re.fullmatch(r'P-\d{3}[a-z]?', i) for i in ids), 'the IDs follow the parity plan, with the F6 record and the CB3 tool tip', [i for i in ids if not re.fullmatch(r'P-\d{3}[a-z]?', i)])
FIELDS = ['id', 'name', 'selector', 'container', 'tab', 'order', 'group', 'text', 'looks', 'kind', 'twips', 'clipped', 'trigger', 'effect', 'note', 'checkedIn']
check(all(set(FIELDS) <= set(r) for r in records) and set(FIELDS) <= set(INVENTORY['fields']), 'every record has every field the inventory explains', [r['id'] for r in records if not set(FIELDS) <= set(r)])
KINDS = {'element', 'menu-entry', 'tooltip', 'state', 'message', 'key', 'behaviour', 'absent'}
check(all(r['kind'] in KINDS and r['looks'] and set(r['looks']) <= {'modern', 'classic'} for r in records), 'every record has a kind and its looks')
selectors = [r['selector'] for r in records if r['selector']] + [r['container'] for r in records if r['container']] + [r['group'] for r in records if r['group']]
bad = page.eval("(list) => list.filter((s) => { try { document.querySelector(s); return false; } catch (e) { return true; } })", selectors)
check(bad == [], 'every selector is valid CSS', bad)
known = {r['selector'] for r in records if r['selector']} | set(WINDOWS)
check(all(r['container'] in known for r in records if r['kind'] in ('element', 'menu-entry', 'tooltip') and r['container']), 'every container is a record or a window', [r['id'] for r in records if r['container'] and r['container'] not in known])

## 14. Every record at its place in the modern look

A fresh page at 1440 by 900 walks every record of the modern look that no other section proves: its
element exists once, lies inside its container, keeps its order among its group and shows its text,
and the effect of a state or a key holds. The main window keeps the caption `3D-Baukasten` with a
field for the name of the scene, the combo writes `Neu_Ende`, and nothing of the first page of the
rewrite is left.

In [ ]:
walker = Browser(site.url, width=1440, height=900, look='modern')
walker.open()
modern = walk(walker, 'modern')
problems = {i: r['problems'] for i, r in modern.items() if r['problems']}
check(problems == {}, 'R-49 every record of the modern look stands at the place 1.3 gives it', problems)
check(len(modern) >= 100, 'the walk covers the records of the modern look', len(modern))
check(walker.texts('#caption') == ['3D-Baukasten'] and walker.visible('#scene-name'), 'Q-2 the main window keeps the caption 3D-Baukasten with a field for the name of the scene', walker.texts('#caption'))
check(walker.eval("() => [...document.querySelectorAll('#cb1 .c-combo-list [data-entry]')].some((e) => e.dataset.entry === 'Neu_Ende')"), 'T-p95b the entry is Neu_Ende')
check(walker.eval("() => document.querySelectorAll('.topbar, .inspector, #inspector, .statusbar, [data-tab=objekte], [data-tab=szenen]').length") == 0, 'R-49 no top bar, inspector or tab of the first page is left')
check(walker.errors() == [], 'no error in the walk', walker.errors())
walker.close()

## 15. Every record at its place in the classic look

A fresh page of the classic look on the virtual screen of 1024 by 768 walks the records of the
classic look the same way, and keeps the place of every element relative to the client of its
container for section 17.

In [ ]:
classic_page = Browser(site.url, width=1024, height=768, look='classic')
classic_page.open(screen='1024x768')
classic = walk(classic_page, 'classic', screen='1024x768', twips=True)
problems = {i: r['problems'] for i, r in classic.items() if r['problems']}
check(problems == {}, 'every record of the classic look stands at its place', problems)
check(len(classic) >= 100, 'the walk covers the records of the classic look', len(classic))
check(classic_page.errors() == [], 'no error in the walk', classic_page.errors())

## 16. The looks share one DOM

The set of ids under the six windows and the set of controls are the same in both looks, the switch
changes the attribute `data-look` of the root and no stylesheet, it creates and removes no element
with an id and no control, and the rows of the list OBJEKTE differ by style alone.

In [ ]:
SNAPSHOT_DOM = "() => ({ ids: [...document.querySelectorAll('#form1 [id], #f2 [id], #f3 [id], #f4 [id], #f5 [id], #f6 [id]')].map((e) => e.id).sort(), controls: [...document.querySelectorAll('button, input, select, textarea')].length, sheets: [...document.styleSheets].filter((s) => !s.disabled).map((s) => s.href || 'inline'), rows: [...document.querySelectorAll('#cb2 [data-line]')].map((r) => r.children.length) })"
OBSERVE = '''() => { window.__mutations = [];
  const keep = (n) => n.nodeType === 1 && (n.id || n.matches('button, input, select, textarea') || n.querySelector('[id], button, input, select, textarea'));
  window.__observer = new MutationObserver((list) => { for (const m of list) for (const n of [...m.addedNodes, ...m.removedNodes]) if (keep(n)) window.__mutations.push(n.id || n.tagName); });
  window.__observer.observe(document.body, { childList: true, subtree: true }); }'''
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
other = 'classic' if look == 'modern' else 'modern'
first = page.eval(SNAPSHOT_DOM)
page.eval(OBSERVE)
check(page.look(other) == other, f'the switch to the {other} look')
page.settle()
second = page.eval(SNAPSHOT_DOM)
mutations = page.eval("() => { window.__observer.disconnect(); return window.__mutations; }")
check(first['ids'] == second['ids'] and len(first['ids']) > 100, 'the ids under the six windows are the same in both looks', sorted(set(first['ids']) ^ set(second['ids'])))
check(first['controls'] == second['controls'], 'the controls are the same', first['controls'], second['controls'])
check(first['sheets'] == second['sheets'] and page.eval("() => document.documentElement.dataset.look") == other, 'the switch changes the attribute data-look and no stylesheet', first['sheets'], second['sheets'])
check(mutations == [], 'the switch creates and removes no element with an id and no control', mutations)
expect(first['rows'] == second['rows'], 'the rows of the list OBJEKTE differ by style alone', first['rows'][:6], second['rows'][:6])
page.look(look)
page.settle()

## 17. Nothing of 1.3 is displaced

Every record with an order keeps it in both looks, and in the classic look at 1024 by 768 every
element with a place of 1.3 in twips lies within 2 px of it, relative to the client of its
container: the frames keep the uneven margins of 1.3. The classic window F4 keeps the register at
245 by 553 px and shows the empty face beyond it when enlarged, while the modern panel follows the
height of the page.

In [ ]:
orders = {i: (modern[i].get('order'), classic[i].get('order')) for i in modern if i in classic and modern[i].get('order') is not None}
check(orders and all(a == b for a, b in orders.values()), 'every record keeps its order in both looks', {i: o for i, o in orders.items() if o[0] != o[1]})
places = {i: r['twips'] for i, r in classic.items() if 'twips' in r}
off = {i: d for i, d in places.items() if abs(d[0]) > 2 or abs(d[1]) > 2}
check(len(places) >= 40 and off == {}, 'F4-Q16 every element lies within 2 px of the twips of 1.3 in the classic look', len(places), off)
frames = {i: places[i] for i in places if i in ('P-110a', 'P-114a', 'P-134')}
check(len(frames) == 3, 'F4-Q16 the frames of tab Körper keep the margins of 1.3', frames)
sstab = classic_page.rect('#sstab1')
check([round(sstab[2]), round(sstab[3])] == [245, 553], 'F4-Q18 the classic register is 245 by 553 px', sstab)
edge = classic_page.rect('#f4 > .win-edge[data-edge="w"]')
classic_page.page.mouse.move(edge[0] + edge[2] / 2, edge[1] + 200); classic_page.page.mouse.down()
classic_page.page.mouse.move(edge[0] - 80, edge[1] + 200, steps=4); classic_page.page.mouse.up(); classic_page.settle()
check(classic_page.rect('#f4')[2] > 252 + 60 and [round(v) for v in classic_page.rect('#sstab1')[2:]] == [245, 553], 'F4-Q18 enlarged, the classic F4 keeps the register at its size and shows the empty face beyond it', classic_page.rect('#f4'), classic_page.rect('#sstab1'))
classic_page.close()
if look == 'modern':
    dock = page.rect('.dock-right')
    check(abs(dock[3] - height) <= 1 and page.eval("() => getComputedStyle(document.querySelector('[data-page=koerper]')).overflowY") in ('auto', 'scroll'), 'F4-Q18 the modern panel follows the height of the page and scrolls its content', dock)

## 18. The new behaviour sits in empty places

The page keeps the four menus of 1.3 and adds Hilfe behind them, keeps thirteen tool places, nine
bodies and six tabs, and puts the field Name and the combo Gruppe into rows 5 and 6 of
GEOMETRIEDATEN, which 1.3 left empty. `#Neu` inserts a declaration group and no CSG-Summe, so the
list reproduces the picture on p. 95 of the thesis line for line.

In [ ]:
counts = page.eval("() => [document.querySelectorAll('#menubar .menu-item').length, document.querySelectorAll('#tb1 .tb-button').length, document.querySelectorAll('#tb2 .tb-button').length, document.querySelectorAll('#sstab1 [role=tab]').length]")
check(counts == [5, 13, 9, 6], 'four menus of 1.3 and Hilfe, thirteen tool places, nine bodies, six tabs', counts)
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
fresh()
page.entry('Quader')
rows = [page.rect(s)[1] for s in ('#t1-7', '#t1-11', '#t1-12')]
check(page.texts('#l1-11') == ['Name'] and page.texts('#l1-12') == ['Gruppe'] and rows[0] < rows[1] < rows[2], 'F4-Q23 row 5 holds the field Name and row 6 the combo Gruppe, under the rotation', rows)
page.entry('#Neu'); page.page.click('#btn-einf'); page.settle()
page.entry('CSG-Differenz'); page.page.click('#btn-einf'); page.settle()
page.page.click('#tb2 [data-body="Quader"]'); page.page.click('#btn-einf'); page.settle()
page.page.click('#tb2 [data-body="Zylinder"]'); page.page.click('#btn-einf'); page.settle()
page.entry('CSG-Ende'); page.page.click('#btn-einf'); page.settle()
page.entry('Neu_Ende'); page.page.click('#btn-einf'); page.settle()
check(page.lines() == ['0 Raster', '1 #Neu Werkstück1', '2 CSG-Differenz', '3   -Quader', '4   -Zylinder', '5 CSG-Ende', '6 Werkstück1'], 'F4-Q50 the list of p. 95 line for line', page.lines())
check(page.eval("() => bausteine.app.scene.items.filter((i) => i.kind === 'MERGE').length") == 0, 'F4-Q50 #Neu inserts no CSG-Summe')

## 19. The report

The report `out/80_parity.md` lists every record with what each look found.
`Documentation/parity.md` is the inventory as a page, which `parity_markdown` of the harness writes
from the fixture, and the notebook checks that the page in the repository equals it. With
`write_docs` the notebook writes the page.

In [ ]:
def plain(text):
    """Answers `text` as one cell of a table of Markdown, its bars escaped and its lines joined."""
    return str(text).replace("|", "\\|").replace("\n", " ")


def verdict(results, record, look_):
    """Answers what the walk of the look `look_` found for `record`, as one cell of the report."""
    if look_ not in record["looks"]:
        return "not in this look"
    if record["checkedIn"]:
        return f"checked in {record['checkedIn']}"
    if record["kind"] in ("behaviour", "absent", "message"):
        return record["kind"]
    found = results.get(record["id"])
    if found is None:
        return "not walked"
    return "ok" if not found["problems"] else plain("; ".join(found["problems"]))


report = ["# The parity walk of notebook 80", "", "| ID | Name | Kind | Modern | Classic |", "| --- | --- | --- | --- | --- |"]
for r in INVENTORY["records"]:
    report.append(f"| {r['id']} | {plain(r['name'])} | `{r['kind']}` | {verdict(modern, r, 'modern')} | {verdict(classic, r, 'classic')} |")
(OUT / "80_parity.md").write_text("\n".join(report) + "\n", encoding="utf-8")
check(all(verdict(modern, r, "modern") != "not walked" and verdict(classic, r, "classic") != "not walked" for r in INVENTORY["records"]), "the report out/80_parity.md names every record in both looks")
DOCS = ROOT / "Documentation" / "parity.md"
# The page comes from the door of the harness, so this check proves the function that writes it.
text = parity_markdown(INVENTORY)
if write_docs:
    DOCS.write_text(text, encoding="utf-8")
check(DOCS.read_text(encoding="utf-8") == text, "Documentation/parity.md is the inventory written by this section")
check(page.errors() == [], "no error on the way", page.errors())

In [ ]:
page.close()
site.stop()
check(DEFERRED == [], 'every check that waited for the end holds', *DEFERRED)